In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import re

/kaggle/input/user-stories-data/user_stories_dataset.csv
/kaggle/input/req-dataset/req_dataset_labeled.csv


# DATASET

In [6]:
from datasets import Dataset, concatenate_datasets


'''
LOAD AND PROCESS DATASET 1 (user stories + acceptance criteria)
'''

file_path = "/kaggle/input/user-stories-data/user_stories_dataset.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError("Ensure your data file exists and Google Drive is mounted.")

# === LOAD DATA ===
df = pd.read_csv(file_path)

# === PREPROCESS LABEL ===
df['ASR'] = df['ASR'].map({"NO": float(0), "YES": float(1)})


# === CLEAN TEXT ===
def clean_text(text):
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9]", " ", text)
    return " ".join(text.lower().split())

# Combine User Story and Acceptance Criteria for the model input
df['user_story'] = df['user_story'].apply(lambda x: clean_text(x))
df['acceptance_criteria'] = df['acceptance_criteria'].apply(lambda x: clean_text(x))
df['text'] = "User Story: " + df['user_story'] + " [SEP] Acceptance Criteria: " + df['acceptance_criteria']

df = df.drop_duplicates().dropna()

# Flatten the labels if they are nested in lists
def flatten_and_int_labels(example):
    # Extract integer if nested in a list/array
    if isinstance(example, list):
        example = example[0]
    
    # Ensure it is a standard integer, not a string or float
    example = int(example)
    return example

df['ASR'] = df['ASR'].map(flatten_and_int_labels)
print(df.head(2))
print(len(df))


dataset1 = Dataset.from_pandas(df[['text', 'ASR']])
print(dataset1[0])
print(dataset1)

                                          user_story  \
0  as a website visitor i want the website to loa...   
1  as a user i want to be able to access the site...   

                                 acceptance_criteria  ASR  \
0  given simultaneous access by multiple users wh...    1   
1  given a smartphone with an ios operating syste...    1   

                                     ASR_explanation FR/NFR  \
0  This story has architectural significance as i...    NFR   
1  The story impacts the overall design by requir...    NFR   

                                                text  
0  User Story: as a website visitor i want the we...  
1  User Story: as a user i want to be able to acc...  
3999
{'text': 'User Story: as a website visitor i want the website to load quickly so that i can quickly access information and have a better user experience [SEP] Acceptance Criteria: given simultaneous access by multiple users when 00 users try to access the website concurrently then the w

In [7]:
'''
LOAD AND PROCESS DATASET 2 (requirement statements)
'''

from datasets import Dataset, concatenate_datasets


file_path2 = '/kaggle/input/req-dataset/req_dataset_labeled.csv'

if not os.path.exists(file_path):
    raise FileNotFoundError("Ensure your data file exists and Google Drive is mounted.")

# === LOAD DATA ===
df2 = pd.read_csv(file_path2)

# === PREPROCESS LABEL ===
df2['ASR'] = df2['ASR'].map({"No": float(0), "Yes": float(1)})

# === CLEAN TEXT ===
def clean_text(text):
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9]", " ", text)
    return " ".join(text.lower().split())

# Combine User Story and Acceptance Criteria for the model input
df2['text'] = df2['requirement'].apply(lambda x: clean_text(x))

df2 = df2.drop_duplicates().dropna()

# Flatten the labels if they are nested in lists
def flatten_and_int_labels(example):
    # Extract integer if nested in a list/array
    if isinstance(example, list):
        example = example[0]
    
    # Ensure it is a standard integer, not a string or float
    example = int(example)
    return example

df2['ASR'] = df2['ASR'].map(flatten_and_int_labels)
print(df2.head(2))
print(len(df2))

dataset2 = Dataset.from_pandas(df2[['text', 'ASR']])
print(dataset2[0])
print(dataset2)


  ID                                        requirement  \
0  0  The system shall provide a graphical user inte...   
1  1  Grid technologies shall be utilized to provide...   

  functional/non-functional              source_file              data_type  \
0                        FR  2004 - grid bgc.pdf.csv  requirement_statement   
1                       NFR  2004 - grid bgc.pdf.csv  requirement_statement   

   ASR                                    ASR_Explanation  \
0    1  This requirement drives the design of the pres...   
1    1  This NFR drives the choice of grid middleware ...   

                                                text  
0  the system shall provide a graphical user inte...  
1  grid technologies shall be utilized to provide...  
6490
{'text': 'the system shall provide a graphical user interface to all functions of the system', 'ASR': 1, '__index_level_0__': 0}
Dataset({
    features: ['text', 'ASR', '__index_level_0__'],
    num_rows: 6490
})


In [8]:
'''
COMBINE DATASETS
'''

from datasets import Dataset, concatenate_datasets


dataset = dataset_cc = concatenate_datasets([dataset1, dataset2])

print(dataset)
print(dataset[0])
print(dataset[-1])

Dataset({
    features: ['text', 'ASR', '__index_level_0__'],
    num_rows: 10489
})
{'text': 'User Story: as a website visitor i want the website to load quickly so that i can quickly access information and have a better user experience [SEP] Acceptance Criteria: given simultaneous access by multiple users when 00 users try to access the website concurrently then the website should be able to handle the load without a significant increase in page load time and it should not crash or show errors 2 given a scenario where the website experiences a sudden surge in trafficwhen the number of concurrent users increases by 50 then the website should be able to scale dynamically to handle the increased load without a drastic decrease in performance', 'ASR': 1, '__index_level_0__': None}
{'text': 'integrate the full text search search tools project module with the warc browser to provide users with warc indexing searching capabilities', 'ASR': 1, '__index_level_0__': 6566}


# TRAINING METRICS

In [9]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    # 1. Per-Class Scores (Returns: [Score_for_Class_0, Score_for_Class_1])
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=None)
    
    # 2. Averages
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(labels, preds, average='micro')
    
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        # Averages
        "f1_macro": macro_f1,
        "f1_micro": micro_f1,
        # "No" (Non-ASR) Specific Scores (Index 0)
        "no_precision": precision[0],
        "no_recall": recall[0],
        "no_f1": f1[0],
        # "Yes" (ASR) Specific Scores (Index 1)
        "yes_precision": precision[1],
        "yes_recall": recall[1],
        "yes_f1": f1[1]
    }

# Clean output directory

In [147]:
!rm -rf /kaggle/working/*

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# ROBERTA (with weighted loss function)

In [10]:
import torch
import torch.nn as nn  # This fixes the NameError
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    RobertaTokenizer, 
    RobertaForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# 1. Load your data
# Replace this with: df = pd.read_csv('your_data.csv')
import pandas as pd
import re


In [94]:
# 1. Custom Trainer for Weighted Loss
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Weights must be on the same device as the model
        self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Remove labels from inputs so the model doesn't compute loss internally
        labels = inputs.pop("labels").long() # Ensure labels are Long tensors
        
        # 2. Get only logits from the model
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 3. Compute your custom weighted CrossEntropyLoss
        # CrossEntropy expects: [Batch, Classes] and [Batch]
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# 2. Setup Data (Assume df contains 'text' and 'label')
# Calculate weights: weight = total_samples / (n_classes * class_samples)
# Example: 1000 samples, 200 ASR (1), 800 Non-ASR (0) -> weights: [0.625, 2.5]
counts = df['ASR'].value_counts().sort_index().values
weights = len(df) / (len(counts) * counts)
print(f"Calculated Class Weights: {weights}")

Calculated Class Weights: [0.85375747 1.20669885]


In [109]:

# 2. Preprocessing
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=False)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.shuffle(seed=42)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

print(tokenized_dataset["train"][0].keys())


Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

dict_keys(['text', 'ASR', '__index_level_0__', 'input_ids', 'attention_mask'])


In [83]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    # 1. Per-Class Scores (Returns: [Score_for_Class_0, Score_for_Class_1])
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=None)
    
    # 2. Averages
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(labels, preds, average='micro')
    
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        # Averages
        "f1_macro": macro_f1,
        "f1_micro": micro_f1,
        # "No" (Non-ASR) Specific Scores (Index 0)
        "no_precision": precision[0],
        "no_recall": recall[0],
        "no_f1": f1[0],
        # "Yes" (ASR) Specific Scores (Index 1)
        "yes_precision": precision[1],
        "yes_recall": recall[1],
        "yes_f1": f1[1]
    }

In [111]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

# 0. CUDA Check & Device Assignment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 4. Training Configuration
model_name = "roberta-base"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

training_args = TrainingArguments(
    num_train_epochs=5,
    # Evaluation Strategy
    eval_strategy="epoch",      # Run eval after every epoch
    logging_strategy="epoch",         # Log metrics after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",       # Use F1 score to determine the best model
    logging_steps=10,             # Prints training loss every 10 steps
    report_to="none",             # Disables external logging (like WandB) to speed up init
    disable_tqdm=False,            # Ensures the progress bar is forced on
    # ... previous args
    learning_rate=2e-5,              # Increased LR
    lr_scheduler_type="cosine",      # Smoother decay than linear
    warmup_ratio=0.1,                # 10% warmup
    weight_decay=0.01,
    per_device_train_batch_size=8,   # Smaller batch can sometimes help escape local minima
    gradient_accumulation_steps=2,   # Keeps effective batch size at 16

)


Using device: cuda


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
# 1. Prepare the dataset specifically for the Trainer
# The Trainer looks for a column named 'labels'

try: 
    tokenized_dataset = tokenized_dataset.rename_column("ASR", "labels")
except:
    print("done already")
# Flatten the labels if they are nested in lists
def flatten_and_int_labels(example):
    # Extract integer if nested in a list/array
    if isinstance(example["labels"], list):
        example["labels"] = example["labels"][0]
    
    # Ensure it is a standard integer, not a string or float
    example["labels"] = int(example["labels"])
    return example

# Apply to your combined dataset
tokenized_dataset = tokenized_dataset.map(flatten_and_int_labels)

print(tokenized_dataset)


done already


Map:   0%|          | 0/8391 [00:00<?, ? examples/s]

Map:   0%|          | 0/2098 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8391
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2098
    })
})


Map:   0%|          | 0/8391 [00:00<?, ? examples/s]

Map:   0%|          | 0/2098 [00:00<?, ? examples/s]

Value('float64')


In [35]:


# 2. Final Trainer Initialization
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer, 
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    class_weights=weights.tolist() 
)



# 3. Execute Training
print("Starting training on CUDA...")
train_results = trainer.train()

# 4. Save the best version
# trainer.save_model("./asr_roberta_final")
print("Model saved to ./asr_roberta_final")

NameError: name 'WeightedTrainer' is not defined

In [114]:
from sklearn.metrics import classification_report

# Get predictions on the evaluation set
predictions = trainer.predict(tokenized_dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Generate and print the report with custom names
report = classification_report(y_true, y_pred, target_names=["No (Non-ASR)", "Yes (ASR)"])
print(report)

              precision    recall  f1-score   support

No (Non-ASR)       0.88      0.81      0.84      1008
   Yes (ASR)       0.83      0.89      0.86      1090

    accuracy                           0.85      2098
   macro avg       0.85      0.85      0.85      2098
weighted avg       0.85      0.85      0.85      2098



# BERT

## BERT all unfrozen

In [77]:
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)

# 0. CUDA Check & Device Assignment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Model & Tokenizer Setup
model_name = "bert-base-uncased"
model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    problem_type="single_label_classification" # Forces CrossEntropyLoss
).to(device)

Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [78]:
# 2. Preprocessing
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.shuffle(seed=42)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

print(tokenized_dataset["train"][0].keys())

try: 
    tokenized_dataset = tokenized_dataset.rename_column("ASR", "labels")
except:
    print("done already")


Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

dict_keys(['text', 'ASR', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])


In [79]:
print(tokenized_dataset)

# Check the unique values in your labels
unique_labels = np.unique(tokenized_dataset["train"]["labels"])
print(f"Unique labels in training set: {unique_labels}")

# Check the test set too
unique_test_labels = np.unique(tokenized_dataset["test"]["labels"])
print(f"Unique labels in test set: {unique_test_labels}")


# 3. Double-check the type (it should say int64)
print(tokenized_dataset["train"].features["labels"])

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8391
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2098
    })
})
Unique labels in training set: [0 1]
Unique labels in test set: [0 1]
Value('int64')


In [84]:
# 3. Basic Training Arguments
training_args = TrainingArguments(
    output_dir="./bert_asr_results",
    num_train_epochs=3,              # Simple 3-epoch run
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    eval_strategy="epoch",      # Eval at end of each epoch
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    save_total_limit=1,               # Save disk space
    report_to="none",             # Disables external logging (like WandB) to speed up init
    disable_tqdm=False,            # Ensures the progress bar is forced on
)

In [85]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

In [86]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,No Precision,No Recall,No F1,Yes Precision,Yes Recall,Yes F1
1,0.246700,0.661140,0.837464,0.836191,0.837464,0.868508,0.779762,0.821746,0.813915,0.890826,0.850635
2,0.309400,0.620038,0.840801,0.839646,0.840801,0.869518,0.786706,0.826042,0.818718,0.890826,0.853251
3,0.086500,0.794793,0.837941,0.836844,0.837941,0.863834,0.786706,0.823468,0.817797,0.885321,0.850220


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=3147, training_loss=0.2137634975401833, metrics={'train_runtime': 581.8206, 'train_samples_per_second': 43.266, 'train_steps_per_second': 5.409, 'total_flos': 988195790609280.0, 'train_loss': 0.2137634975401833, 'epoch': 3.0})

In [87]:
from sklearn.metrics import classification_report

# Get predictions on the evaluation set
predictions = trainer.predict(tokenized_dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Generate and print the report with custom names
report = classification_report(y_true, y_pred, target_names=["No (Non-ASR)", "Yes (ASR)"])
print(report)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


              precision    recall  f1-score   support

No (Non-ASR)       0.87      0.79      0.83      1008
   Yes (ASR)       0.82      0.89      0.85      1090

    accuracy                           0.84      2098
   macro avg       0.84      0.84      0.84      2098
weighted avg       0.84      0.84      0.84      2098



## BERT with all layers frozen except the classification head

In [94]:
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)

# 0. CUDA Check & Device Assignment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Model & Tokenizer Setup
model_name = "bert-base-uncased"
model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    problem_type="single_label_classification" # Forces CrossEntropyLoss
).to(device)

for name, param in model.base_model.named_parameters():
    param.requires_grad = False

# 3. Basic Training Arguments
training_args = TrainingArguments(
    num_train_epochs=3,              # Simple 3-epoch run
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    eval_strategy="epoch",      # Eval at end of each epoch
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    save_total_limit=0,               # Save disk space
    report_to="none",             # Disables external logging (like WandB) to speed up init
    disable_tqdm=False,            # Ensures the progress bar is forced on
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using device: cuda


In [95]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,No Precision,No Recall,No F1,Yes Precision,Yes Recall,Yes F1
1,0.665400,0.661626,0.676358,0.659789,0.676358,0.762360,0.474206,0.584709,0.639701,0.863303,0.734869
2,0.680400,0.653026,0.665872,0.651574,0.665872,0.730827,0.482143,0.580992,0.635729,0.835780,0.722156
3,0.665800,0.650827,0.662059,0.648159,0.662059,0.722140,0.482143,0.578227,0.633684,0.828440,0.718091


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=3147, training_loss=0.6708024342854818, metrics={'train_runtime': 399.2824, 'train_samples_per_second': 63.046, 'train_steps_per_second': 7.882, 'total_flos': 988195790609280.0, 'train_loss': 0.6708024342854818, 'epoch': 3.0})

In [ ]:
from sklearn.metrics import classification_report

# Get predictions on the evaluation set
predictions = trainer.predict(tokenized_dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Generate and print the report with custom names
report = classification_report(y_true, y_pred, target_names=["No (Non-ASR)", "Yes (ASR)"])
print(report)

## BERT with pooling layers unfrozen

In [96]:
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)

# 0. CUDA Check & Device Assignment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Model & Tokenizer Setup
model_name = "bert-base-uncased"
model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    problem_type="single_label_classification" # Forces CrossEntropyLoss
).to(device)

for name, param in model.base_model.named_parameters():
    param.requires_grad = False

for name, param in model.base_model.named_parameters():
    if 'pooler' in name:
        param.requires_grad = True

# 3. Basic Training Arguments
training_args = TrainingArguments(
    num_train_epochs=3,              # Simple 3-epoch run
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    eval_strategy="epoch",      # Eval at end of each epoch
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    save_total_limit=0,               # Save disk space
    report_to="none",             # Disables external logging (like WandB) to speed up init
    disable_tqdm=False,            # Ensures the progress bar is forced on
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [97]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,No Precision,No Recall,No F1,Yes Precision,Yes Recall,Yes F1
1,0.542000,0.513229,0.763584,0.758096,0.763584,0.830749,0.637897,0.721661,0.724320,0.879817,0.794532
2,0.561900,0.491157,0.756911,0.752630,0.756911,0.805897,0.650794,0.720088,0.725857,0.855046,0.785173
3,0.509000,0.481519,0.762154,0.758756,0.762154,0.802616,0.669643,0.730124,0.735084,0.847706,0.787388


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=3147, training_loss=0.5309745957065772, metrics={'train_runtime': 403.1916, 'train_samples_per_second': 62.434, 'train_steps_per_second': 7.805, 'total_flos': 988195790609280.0, 'train_loss': 0.5309745957065772, 'epoch': 3.0})

In [98]:
from sklearn.metrics import classification_report

# Get predictions on the evaluation set
predictions = trainer.predict(tokenized_dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Generate and print the report with custom names
report = classification_report(y_true, y_pred, target_names=["No (Non-ASR)", "Yes (ASR)"])
print(report)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


              precision    recall  f1-score   support

No (Non-ASR)       0.83      0.64      0.72      1008
   Yes (ASR)       0.72      0.88      0.79      1090

    accuracy                           0.76      2098
   macro avg       0.78      0.76      0.76      2098
weighted avg       0.78      0.76      0.76      2098



## BERT full unlocked + weighted loss function

In [99]:
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)

# 0. CUDA Check & Device Assignment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Model & Tokenizer Setup
model_name = "bert-base-uncased"
model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    problem_type="single_label_classification" # Forces CrossEntropyLoss
).to(device)

# 2. Preprocessing
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.shuffle(seed=42)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

print(tokenized_dataset["train"][0].keys())

try: 
    tokenized_dataset = tokenized_dataset.rename_column("ASR", "labels")
except:
    print("done already")

print(tokenized_dataset)

# Check the unique values in your labels
unique_labels = np.unique(tokenized_dataset["train"]["labels"])
print(f"Unique labels in training set: {unique_labels}")

# Check the test set too
unique_test_labels = np.unique(tokenized_dataset["test"]["labels"])
print(f"Unique labels in test set: {unique_test_labels}")


# 3. Double-check the type (it should say int64)
print(tokenized_dataset["train"].features["labels"])



Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

dict_keys(['text', 'ASR', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8391
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2098
    })
})
Unique labels in training set: [0 1]
Unique labels in test set: [0 1]
Value('int64')
Calculated Class Weights: [0.85375747 1.20669885]


In [107]:
# 1. Custom Trainer for Weighted Loss
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Weights must be on the same device as the model
        self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Remove labels from inputs so the model doesn't compute loss internally
        labels = inputs.pop("labels").long() # Ensure labels are Long tensors
        
        # 2. Get only logits from the model
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 3. Compute your custom weighted CrossEntropyLoss
        # CrossEntropy expects: [Batch, Classes] and [Batch]
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# 2. Setup Data (Assume df contains 'text' and 'label')
# Calculate weights: weight = total_samples / (n_classes * class_samples)
# Example: 1000 samples, 200 ASR (1), 800 Non-ASR (0) -> weights: [0.625, 2.5]
counts = df['ASR'].value_counts().sort_index().values
weights = len(df) / (len(counts) * counts)
print(f"Calculated Class Weights: {weights}")

training_args = TrainingArguments(
    num_train_epochs=10,
    # Evaluation Strategy
    eval_strategy="epoch",      # Run eval after every epoch
    logging_strategy="epoch",         # Log metrics after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",       # Use F1 score to determine the best model
    logging_steps=10,             # Prints training loss every 10 steps
    report_to="none",             # Disables external logging (like WandB) to speed up init
    disable_tqdm=False,            # Ensures the progress bar is forced on
    # ... previous args
    learning_rate=2e-5,              # Increased LR
    lr_scheduler_type="cosine",      # Smoother decay than linear
    warmup_ratio=0.1,                # 10% warmup
    weight_decay=0.01,
    per_device_train_batch_size=8,   # Smaller batch can sometimes help escape local minima
    gradient_accumulation_steps=2,   # Keeps effective batch size at 16

)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer, 
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    class_weights=weights.tolist() 
)

Calculated Class Weights: [0.85375747 1.20669885]


In [108]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,No Precision,No Recall,No F1,Yes Precision,Yes Recall,Yes F1
1,0.064000,0.641208,0.846044,0.844285,0.846044,0.895040,0.769841,0.827733,0.811535,0.916514,0.860836
2,0.107000,0.633632,0.836034,0.835872,0.836034,0.824219,0.837302,0.830709,0.847300,0.834862,0.841035
3,0.077900,0.774391,0.833651,0.830656,0.833651,0.906289,0.729167,0.808136,0.787879,0.930275,0.853176
4,0.084600,0.761443,0.844137,0.843856,0.844137,0.840160,0.834325,0.837232,0.847767,0.853211,0.850480
5,0.070800,0.742632,0.844614,0.843356,0.844614,0.878049,0.785714,0.829319,0.819398,0.899083,0.857393
6,0.063700,0.788737,0.851764,0.851091,0.851764,0.867229,0.816468,0.841083,0.838990,0.884404,0.861099
7,0.044300,0.866194,0.850810,0.850541,0.850810,0.847153,0.841270,0.844201,0.854148,0.859633,0.856882
8,0.036200,0.859479,0.853670,0.853006,0.853670,0.869336,0.818452,0.843127,0.840731,0.886239,0.862885
9,0.036000,0.879640,0.846997,0.845772,0.846997,0.880399,0.788690,0.832025,0.821757,0.900917,0.859519
10,0.031000,0.875535,0.851287,0.850304,0.851287,0.878261,0.801587,0.838174,0.830221,0.897248,0.862434


TrainOutput(global_step=2630, training_loss=0.06155204954256123, metrics={'train_runtime': 1313.2846, 'train_samples_per_second': 63.893, 'train_steps_per_second': 2.003, 'total_flos': 3750313552561020.0, 'train_loss': 0.06155204954256123, 'epoch': 10.0})

In [109]:
from sklearn.metrics import classification_report

# Get predictions on the evaluation set
predictions = trainer.predict(tokenized_dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Generate and print the report with custom names
report = classification_report(y_true, y_pred, target_names=["No (Non-ASR)", "Yes (ASR)"])
print(report)

              precision    recall  f1-score   support

No (Non-ASR)       0.87      0.82      0.84      1008
   Yes (ASR)       0.84      0.89      0.86      1090

    accuracy                           0.85      2098
   macro avg       0.86      0.85      0.85      2098
weighted avg       0.85      0.85      0.85      2098



# DistillBERT w/ weighted trainer, unlocked

In [110]:
from transformers import (
    DistilBertTokenizer, 
    DistilBertForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)

# 0. CUDA Check & Device Assignment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Model & Tokenizer Setup
model_name = "bert-base-uncased"
model = DistilBertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    problem_type="single_label_classification" # Forces CrossEntropyLoss
).to(device)

# 2. Preprocessing
model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.shuffle(seed=42)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

print(tokenized_dataset["train"][0].keys())

try: 
    tokenized_dataset = tokenized_dataset.rename_column("ASR", "labels")
except:
    print("done already")

print(tokenized_dataset)

# Check the unique values in your labels
unique_labels = np.unique(tokenized_dataset["train"]["labels"])
print(f"Unique labels in training set: {unique_labels}")

# Check the test set too
unique_test_labels = np.unique(tokenized_dataset["test"]["labels"])
print(f"Unique labels in test set: {unique_test_labels}")


# 3. Double-check the type (it should say int64)
print(tokenized_dataset["train"].features["labels"])



Using device: cuda


You are using a model of type bert to instantiate a model of type distilbert. This is not supported for all configurations of models and can yield errors.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'distilbert.embeddings.LayerNorm.bias', 'distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.position_embeddings.weight', 'distilbert.embeddings.word_embeddings.weight', 'distilbert.transformer.layer.0.attention.k_lin.bias', 'distilbert.transformer.layer.0.attention.k_lin.weight', 'distilbert.transformer.layer.0.attention.out_lin.bias', 'distilbert.transformer.layer.0.attention.out_lin.weight', 'distilbert.transformer.layer.0.attention.q_lin.bias', 'distilbert.transformer.layer.0.attention.q_lin.weight', 'distilbert.transformer.layer.0.attention.v_lin.bias', 'distilbert.transformer.layer.0.attention.v_lin.weight', 'distilbert.transformer

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

dict_keys(['text', 'ASR', '__index_level_0__', 'input_ids', 'attention_mask'])
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask'],
        num_rows: 8391
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'attention_mask'],
        num_rows: 2098
    })
})
Unique labels in training set: [0 1]
Unique labels in test set: [0 1]
Value('int64')


In [111]:
# 1. Custom Trainer for Weighted Loss
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Weights must be on the same device as the model
        self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Remove labels from inputs so the model doesn't compute loss internally
        labels = inputs.pop("labels").long() # Ensure labels are Long tensors
        
        # 2. Get only logits from the model
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 3. Compute your custom weighted CrossEntropyLoss
        # CrossEntropy expects: [Batch, Classes] and [Batch]
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# 2. Setup Data (Assume df contains 'text' and 'label')
# Calculate weights: weight = total_samples / (n_classes * class_samples)
# Example: 1000 samples, 200 ASR (1), 800 Non-ASR (0) -> weights: [0.625, 2.5]
counts = df['ASR'].value_counts().sort_index().values
weights = len(df) / (len(counts) * counts)
print(f"Calculated Class Weights: {weights}")

training_args = TrainingArguments(
    num_train_epochs=10,
    # Evaluation Strategy
    eval_strategy="epoch",      # Run eval after every epoch
    logging_strategy="epoch",         # Log metrics after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",       # Use F1 score to determine the best model
    logging_steps=10,             # Prints training loss every 10 steps
    report_to="none",             # Disables external logging (like WandB) to speed up init
    disable_tqdm=False,            # Ensures the progress bar is forced on
    # ... previous args
    learning_rate=2e-5,              # Increased LR
    lr_scheduler_type="cosine",      # Smoother decay than linear
    warmup_ratio=0.1,                # 10% warmup
    weight_decay=0.01,
    per_device_train_batch_size=8,   # Smaller batch can sometimes help escape local minima
    gradient_accumulation_steps=2,   # Keeps effective batch size at 16

)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer, 
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    class_weights=weights.tolist() 
)

Calculated Class Weights: [0.85375747 1.20669885]


In [112]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,No Precision,No Recall,No F1,Yes Precision,Yes Recall,Yes F1
1,0.585600,0.462037,0.733556,0.712880,0.733556,0.925996,0.484127,0.635831,0.669001,0.964220,0.789929
2,0.437000,0.395759,0.809342,0.808052,0.809342,0.831155,0.756944,0.792316,0.792373,0.857798,0.823789
3,0.354100,0.378957,0.801716,0.798224,0.801716,0.863636,0.697421,0.771679,0.762461,0.898165,0.824768
4,0.296200,0.396089,0.813632,0.811579,0.813632,0.854191,0.738095,0.791911,0.784841,0.883486,0.831247
5,0.252400,0.444919,0.811725,0.808084,0.811725,0.882647,0.701389,0.781647,0.767926,0.913761,0.834520
6,0.220700,0.432407,0.791706,0.786257,0.791706,0.878146,0.657738,0.752127,0.743112,0.915596,0.820386
7,0.192600,0.530961,0.827931,0.826808,0.827931,0.851249,0.777778,0.812856,0.809686,0.874312,0.840759
8,0.174000,0.501621,0.832221,0.831670,0.832221,0.838144,0.806548,0.822042,0.827128,0.855963,0.841298
9,0.154100,0.541489,0.819828,0.818109,0.819828,0.855530,0.751984,0.800422,0.793729,0.882569,0.835795
10,0.146300,0.534661,0.822688,0.821159,0.822688,0.854911,0.759921,0.804622,0.798669,0.880734,0.837696


TrainOutput(global_step=2630, training_loss=0.2813051956234776, metrics={'train_runtime': 1303.7018, 'train_samples_per_second': 64.363, 'train_steps_per_second': 2.017, 'total_flos': 3750313552561020.0, 'train_loss': 0.2813051956234776, 'epoch': 10.0})

In [113]:
from sklearn.metrics import classification_report

# Get predictions on the evaluation set
predictions = trainer.predict(tokenized_dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Generate and print the report with custom names
report = classification_report(y_true, y_pred, target_names=["No (Non-ASR)", "Yes (ASR)"])
print(report)

              precision    recall  f1-score   support

No (Non-ASR)       0.84      0.81      0.82      1008
   Yes (ASR)       0.83      0.86      0.84      1090

    accuracy                           0.83      2098
   macro avg       0.83      0.83      0.83      2098
weighted avg       0.83      0.83      0.83      2098



# DeBERTaV3

In [148]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)

# 1. Setup DeBERTa-v3-base
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    problem_type="single_label_classification" # Solves the target size error
).to(device)


def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.shuffle(seed=42)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

print(tokenized_dataset["train"][0].keys())

try: 
    tokenized_dataset = tokenized_dataset.rename_column("ASR", "labels")
except:
    print("done already")

print(tokenized_dataset)

# Check the unique values in your labels
unique_labels = np.unique(tokenized_dataset["train"]["labels"])
print(f"Unique labels in training set: {unique_labels}")

# Check the test set too
unique_test_labels = np.unique(tokenized_dataset["test"]["labels"])
print(f"Unique labels in test set: {unique_test_labels}")


# 3. Double-check the type (it should say int64)
print(tokenized_dataset["train"].features["labels"])



/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


dict_keys(['text', 'ASR', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8391
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2098
    })
})
Unique labels in training set: [0 1]
Unique labels in test set: [0 1]
Value('int64')


In [150]:
# 1. Custom Trainer for Weighted Loss
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Weights must be on the same device as the model
        self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Remove labels from inputs so the model doesn't compute loss internally
        labels = inputs.pop("labels").long() # Ensure labels are Long tensors
        
        # 2. Get only logits from the model
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 3. Compute your custom weighted CrossEntropyLoss
        # CrossEntropy expects: [Batch, Classes] and [Batch]
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# 2. Setup Data (Assume df contains 'text' and 'label')
# Calculate weights: weight = total_samples / (n_classes * class_samples)
# Example: 1000 samples, 200 ASR (1), 800 Non-ASR (0) -> weights: [0.625, 2.5]
counts = df['ASR'].value_counts().sort_index().values
weights = len(df) / (len(counts) * counts)
print(f"Calculated Class Weights: {weights}")

training_args = TrainingArguments(
    num_train_epochs=10,
    # Evaluation Strategy
    eval_strategy="epoch",      # Run eval after every epoch
    logging_strategy="epoch",         # Log metrics after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",       # Use F1 score to determine the best model
    logging_steps=10,             # Prints training loss every 10 steps
    report_to="none",             # Disables external logging (like WandB) to speed up init
    disable_tqdm=False,            # Ensures the progress bar is forced on
    # ... previous args
    learning_rate=2e-5,              # Increased LR
    lr_scheduler_type="cosine",      # Smoother decay than linear
    warmup_ratio=0.1,                # 10% warmup
    weight_decay=0.01,
    per_device_train_batch_size=8,   # Smaller batch can sometimes help escape local minima
    gradient_accumulation_steps=2,   # Keeps effective batch size at 16
save_total_limit=1,              # Keep ONLY the best/last checkpoint
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer, 
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    class_weights=weights.tolist() 
)

Calculated Class Weights: [0.85375747 1.20669885]


In [151]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,No Precision,No Recall,No F1,Yes Precision,Yes Recall,Yes F1
1,0.530500,0.367382,0.820305,0.817164,0.820305,0.887117,0.717262,0.793198,0.777864,0.915596,0.841129
2,0.352900,0.339324,0.841754,0.840153,0.841754,0.884091,0.771825,0.824153,0.811166,0.906422,0.856153
3,0.251200,0.348820,0.846044,0.845423,0.846044,0.857889,0.814484,0.835623,0.836109,0.875229,0.855222
4,0.178300,0.464032,0.844137,0.844038,0.844137,0.828351,0.852183,0.840098,0.859566,0.836697,0.847978
5,0.130200,0.514318,0.844614,0.841877,0.844614,0.918919,0.742063,0.821076,0.797508,0.939450,0.862679
6,0.098700,0.573433,0.854147,0.852836,0.854147,0.893498,0.790675,0.838947,0.825041,0.912844,0.866725
7,0.070100,0.667457,0.852717,0.851380,0.852717,0.892256,0.788690,0.837283,0.823529,0.911927,0.865477
8,0.055900,0.751238,0.854147,0.853160,0.854147,0.882353,0.803571,0.841121,0.832203,0.900917,0.865198
9,0.046500,0.774586,0.854623,0.853866,0.854623,0.874334,0.814484,0.843349,0.838654,0.891743,0.864384
10,0.043900,0.777426,0.854623,0.853826,0.854623,0.875936,0.812500,0.843026,0.837489,0.893578,0.864625


TrainOutput(global_step=2630, training_loss=0.1758114996065205, metrics={'train_runtime': 1974.0319, 'train_samples_per_second': 42.507, 'train_steps_per_second': 1.332, 'total_flos': 3607313663595948.0, 'train_loss': 0.1758114996065205, 'epoch': 10.0})

In [152]:
from sklearn.metrics import classification_report

# Get predictions on the evaluation set
predictions = trainer.predict(tokenized_dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Generate and print the report with customnames
report = classification_report(y_true, y_pred, target_names=["No (Non-ASR)", "Yes (ASR)"])
print(report)

              precision    recall  f1-score   support

No (Non-ASR)       0.87      0.81      0.84      1008
   Yes (ASR)       0.84      0.89      0.86      1090

    accuracy                           0.85      2098
   macro avg       0.86      0.85      0.85      2098
weighted avg       0.86      0.85      0.85      2098



# TinyBERT

In [153]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)

# 1. Load TinyBERT Tokenizer and Model
# Note: TinyBERT uses the BERT tokenizer logic
model_name = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    problem_type="single_label_classification"
).to("cuda")


def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.shuffle(seed=42)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

print(tokenized_dataset["train"][0].keys())

try: 
    tokenized_dataset = tokenized_dataset.rename_column("ASR", "labels")
except:
    print("done already")

print(tokenized_dataset)

# Check the unique values in your labels
unique_labels = np.unique(tokenized_dataset["train"]["labels"])
print(f"Unique labels in training set: {unique_labels}")

# Check the test set too
unique_test_labels = np.unique(tokenized_dataset["test"]["labels"])
print(f"Unique labels in test set: {unique_test_labels}")


# 3. Double-check the type (it should say int64)
print(tokenized_dataset["train"].features["labels"])



config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


model.safetensors:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

dict_keys(['text', 'ASR', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8391
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2098
    })
})
Unique labels in training set: [0 1]
Unique labels in test set: [0 1]
Value('int64')


In [154]:
# 1. Custom Trainer for Weighted Loss
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Weights must be on the same device as the model
        self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Remove labels from inputs so the model doesn't compute loss internally
        labels = inputs.pop("labels").long() # Ensure labels are Long tensors
        
        # 2. Get only logits from the model
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 3. Compute your custom weighted CrossEntropyLoss
        # CrossEntropy expects: [Batch, Classes] and [Batch]
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# 2. Setup Data (Assume df contains 'text' and 'label')
# Calculate weights: weight = total_samples / (n_classes * class_samples)
# Example: 1000 samples, 200 ASR (1), 800 Non-ASR (0) -> weights: [0.625, 2.5]
counts = df['ASR'].value_counts().sort_index().values
weights = len(df) / (len(counts) * counts)
print(f"Calculated Class Weights: {weights}")

training_args = TrainingArguments(
    num_train_epochs=10,
    # Evaluation Strategy
    eval_strategy="epoch",      # Run eval after every epoch
    logging_strategy="epoch",         # Log metrics after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",       # Use F1 score to determine the best model
    logging_steps=10,             # Prints training loss every 10 steps
    report_to="none",             # Disables external logging (like WandB) to speed up init
    disable_tqdm=False,            # Ensures the progress bar is forced on
    # ... previous args
    learning_rate=2e-5,              # Increased LR
    lr_scheduler_type="cosine",      # Smoother decay than linear
    warmup_ratio=0.1,                # 10% warmup
    weight_decay=0.01,
    per_device_train_batch_size=8,   # Smaller batch can sometimes help escape local minima
    gradient_accumulation_steps=2,   # Keeps effective batch size at 16
save_total_limit=1,              # Keep ONLY the best/last checkpoint
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer, 
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    class_weights=weights.tolist() 
)

Calculated Class Weights: [0.85375747 1.20669885]


In [155]:

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,No Precision,No Recall,No F1,Yes Precision,Yes Recall,Yes F1
1,0.606900,0.469978,0.779790,0.775379,0.779790,0.842965,0.665675,0.743902,0.741167,0.885321,0.806856
2,0.443200,0.431795,0.785510,0.777250,0.785510,0.906706,0.617063,0.734357,0.726629,0.941284,0.820144
3,0.394700,0.420316,0.794090,0.786483,0.794090,0.914986,0.629960,0.746181,0.734330,0.945872,0.826784
4,0.349000,0.429730,0.794090,0.785996,0.794090,0.922287,0.624008,0.744379,0.732345,0.951376,0.827614
5,0.308300,0.404628,0.810772,0.805450,0.810772,0.911171,0.671627,0.773272,0.755720,0.939450,0.837628
6,0.286900,0.430608,0.803146,0.796286,0.803146,0.921986,0.644841,0.758903,0.743001,0.949541,0.833669
7,0.253900,0.447264,0.800286,0.792869,0.800286,0.924964,0.635913,0.753674,0.738790,0.952294,0.832064
8,0.239600,0.438336,0.806482,0.800408,0.806482,0.915746,0.657738,0.765589,0.748908,0.944037,0.835227
9,0.232600,0.451313,0.804576,0.797655,0.804576,0.925926,0.644841,0.760234,0.743553,0.952294,0.835076
10,0.223900,0.438698,0.812679,0.807157,0.812679,0.918367,0.669643,0.774527,0.755686,0.944954,0.839788


TrainOutput(global_step=2630, training_loss=0.3339151244653042, metrics={'train_runtime': 202.1027, 'train_samples_per_second': 415.185, 'train_steps_per_second': 13.013, 'total_flos': 204384159363468.0, 'train_loss': 0.3339151244653042, 'epoch': 10.0})

In [156]:
from sklearn.metrics import classification_report

# Get predictions on the evaluation set
predictions = trainer.predict(tokenized_dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Generate and print the report with customnames
report = classification_report(y_true, y_pred, target_names=["No (Non-ASR)", "Yes (ASR)"])
print(report)

              precision    recall  f1-score   support

No (Non-ASR)       0.92      0.67      0.77      1008
   Yes (ASR)       0.76      0.94      0.84      1090

    accuracy                           0.81      2098
   macro avg       0.84      0.81      0.81      2098
weighted avg       0.83      0.81      0.81      2098



# XLNET

In [10]:
import torch
import numpy as np  # Added for the np.unique calls later
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)

# 1. Load XLNet Tokenizer and Model
# Replaced TinyBERT with XLNet (Base Cased version)
model_name = "xlnet-base-cased" 
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    problem_type="single_label_classification"
).to("cuda")


def tokenize_function(examples):
    # AutoTokenizer for XLNet automatically handles its specific requirements
    # (Note: XLNet defaults to padding on the left, unlike BERT)
    return tokenizer(examples["text"], truncation=True)

# Assumes 'dataset' is already loaded in your environment
tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.shuffle(seed=42)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

print(tokenized_dataset["train"][0].keys())

try: 
    tokenized_dataset = tokenized_dataset.rename_column("ASR", "labels")
except:
    print("done already")

print(tokenized_dataset)

# Check the unique values in your labels
unique_labels = np.unique(tokenized_dataset["train"]["labels"])
print(f"Unique labels in training set: {unique_labels}")

# Check the test set too
unique_test_labels = np.unique(tokenized_dataset["test"]["labels"])
print(f"Unique labels in test set: {unique_test_labels}")


# 3. Double-check the type (it should say int64)
print(tokenized_dataset["train"].features["labels"])

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/467M [00:00<?, ?B/s]

Some weights of XLNetForSequenceClassification were not initialized from the model checkpoint at xlnet-base-cased and are newly initialized: ['logits_proj.bias', 'logits_proj.weight', 'sequence_summary.summary.bias', 'sequence_summary.summary.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/10489 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


model.safetensors:   0%|          | 0.00/467M [00:00<?, ?B/s]

dict_keys(['text', 'ASR', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'])
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8391
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2098
    })
})
Unique labels in training set: [0 1]
Unique labels in test set: [0 1]
Value('int64')


In [11]:
import torch
import torch.nn as nn
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

# 1. Custom Trainer for Weighted Loss
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Weights must be on the same device as the model
        # We use .to(self.args.device) to ensure it matches the trainer's device configuration
        self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 1. Remove labels from inputs so the model doesn't compute loss internally
        # XLNet's forward pass accepts labels, but we want to calculate our own loss
        labels = inputs.pop("labels").long() 
        
        # 2. Get only logits from the model
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 3. Compute your custom weighted CrossEntropyLoss
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# 2. Setup Data 
# (Assuming 'df' and 'tokenized_dataset' are defined from previous steps)

# Calculate weights based on your 'ASR' column
counts = df['ASR'].value_counts().sort_index().values
weights = len(df) / (len(counts) * counts)
print(f"Calculated Class Weights: {weights}")

training_args = TrainingArguments(
    output_dir="./xlnet_results",      # Changed output dir name for clarity
    num_train_epochs=10,
    eval_strategy="epoch",             # Run eval after every epoch
    logging_strategy="epoch",          # Log metrics after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",  # Ensure you have 'f1_micro' in your compute_metrics
    logging_steps=10,
    report_to="none",
    disable_tqdm=False,
    
    # Hyperparameters
    learning_rate=2e-5,                # XLNet is sensitive; 2e-5 is a safe starting point
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    per_device_train_batch_size=8,     # XLNet is larger than TinyBERT; watch out for OOM errors
    gradient_accumulation_steps=2,
    save_total_limit=1,
)

trainer = WeightedTrainer(
    model=model,                       # This is your XLNet model from step 1
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,        # Uses the XLNet tokenizer
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,   # Ensure this function is defined
    class_weights=weights.tolist() 
)

Calculated Class Weights: [0.85375747 1.20669885]


In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,No Precision,No Recall,No F1,Yes Precision,Yes Recall,Yes F1
1,1.049400,0.396303,0.814109,0.814060,0.814109,0.780784,0.843750,0.811047,0.848928,0.787523,0.817073
2,0.779500,0.347737,0.832698,0.828687,0.832698,0.908280,0.718750,0.802476,0.787510,0.934901,0.854899
3,0.626800,0.357115,0.840324,0.839493,0.840324,0.843979,0.812500,0.827940,0.837270,0.865280,0.851045
4,0.476900,0.390141,0.835081,0.834643,0.835081,0.823647,0.828629,0.826131,0.845455,0.840868,0.843155
5,0.332900,0.479143,0.845091,0.843320,0.845091,0.877690,0.781250,0.826667,0.821399,0.902351,0.859974
6,0.245300,0.571212,0.840801,0.840438,0.840801,0.827038,0.838710,0.832833,0.853480,0.842676,0.848044
7,0.180800,0.684127,0.850334,0.849502,0.850334,0.856842,0.820565,0.838311,0.844948,0.877034,0.860692
8,0.143900,0.788847,0.845567,0.844709,0.845567,0.851579,0.815524,0.833162,0.840592,0.872514,0.856256
9,0.124500,0.834220,0.843661,0.842921,0.843661,0.845114,0.819556,0.832139,0.842430,0.865280,0.853702
10,0.115800,0.835055,0.846520,0.845645,0.846520,0.853376,0.815524,0.834021,0.840870,0.874322,0.857270


TrainOutput(global_step=2630, training_loss=0.4075948287325667, metrics={'train_runtime': 1733.5018, 'train_samples_per_second': 48.405, 'train_steps_per_second': 1.517, 'total_flos': 4443534223705500.0, 'train_loss': 0.4075948287325667, 'epoch': 10.0})

In [13]:
from sklearn.metrics import classification_report

# Get predictions on the evaluation set
predictions = trainer.predict(tokenized_dataset["test"])
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(-1)

# Generate and print the report with customnames
report = classification_report(y_true, y_pred, target_names=["No (Non-ASR)", "Yes (ASR)"])
print(report)

              precision    recall  f1-score   support

No (Non-ASR)       0.86      0.82      0.84       992
   Yes (ASR)       0.84      0.88      0.86      1106

    accuracy                           0.85      2098
   macro avg       0.85      0.85      0.85      2098
weighted avg       0.85      0.85      0.85      2098

